# ASSIGNMENT 1 - Question 9 - predictive Maintenance of Wind Turbines
### Alumno: Martín Andrés Gutiérrez López
Topics # ASSIGNMENT 1 - Question 8 - Hourly Electricity-Load Forecasting
### Alumno: Martín Andrés Gutiérrez López
Topics vanishing/exploding gradients, LSTM gates & cell state, GRU mechanics, gradient clipping (value/norm)

In [1]:
# ==============================================================================
# Question 9: Predictive Maintenance of Wind Turbines (Case: AeroWind)
# LSTM/GRU Transitions, Parameter Counts, and Gradient Clipping Mechanics
# ==============================================================================

import numpy as np
import torch
import torch.nn as nn

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ------------------------------------------------------------------------------
# Part (a): LSTM Cell State and Hidden State Computation
# ------------------------------------------------------------------------------
f = 0.9  # Forget gate
i = 0.2  # Input gate
g = 0.5  # Candidate cell state
o = 0.7  # Output gate
c_prev = 2.0  # c_{t-1}

c_t = f * c_prev + i * g
h_t_lstm = o * np.tanh(c_t)

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (a): LSTM COMPUTATION AT TIME STEP t")
print("=" * 75,f"{RESET}")

print(f"New Cell State (c_t)   : {c_t:.4f} (Expected: 1.9000)")
print(f"New Hidden State (h_t) : {h_t_lstm:.6f} (Expected: 0.668971)")
print()

# ------------------------------------------------------------------------------
# Part (b): GRU Hidden State Computation
# ------------------------------------------------------------------------------
z = 0.3  # Update gate
h_prev = 0.8  # h_{t-1}
h_tilde = -0.4  # Candidate hidden state

h_t_gru = (1.0 - z) * h_prev + z * h_tilde

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (b): GRU COMPUTATION AT TIME STEP t")
print("=" * 75,f"{RESET}")
print(f"New GRU Hidden State (h_t) : {h_t_gru:.4f} (Expected: 0.4400)")
print()

# ------------------------------------------------------------------------------
# Part (c): Parameter Audit (Input = 8, Hidden = 32, one bias vector per gate)
# ------------------------------------------------------------------------------
input_size = 8
hidden_size = 32

# Analytical parameter formula: num_gates * ((input_size * hidden_size) + (hidden_size * hidden_size) + hidden_size)
params_per_gate = (
    (input_size * hidden_size) + (hidden_size * hidden_size) + hidden_size
)
lstm_params_analytical = 4 * params_per_gate
gru_params_analytical = 3 * params_per_gate

# PyTorch verification (PyTorch uses 2 bias vectors per gate, so we subtract redundant biases to match problem specification)
lstm_layer = nn.LSTM(input_size, hidden_size, bias=True)
gru_layer = nn.GRU(input_size, hidden_size, bias=True)

# Count parameters matching single-bias specification
lstm_single_bias_params = (
    lstm_layer.weight_ih_l0.numel()
    + lstm_layer.weight_hh_l0.numel()
    + lstm_layer.bias_ih_l0.numel()
)
gru_single_bias_params = (
    gru_layer.weight_ih_l0.numel()
    + gru_layer.weight_hh_l0.numel()
    + gru_layer.bias_ih_l0.numel()
)

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (c): PARAMETER COUNTS (INPUT 8, HIDDEN 32)")
print("=" * 75,f"{RESET}")
print(f"Parameters per gate component : {params_per_gate}")
print(
    f"Total LSTM Parameters (4 gates): {lstm_params_analytical} (PyTorch"
    f" verified: {lstm_single_bias_params})"
)
print(
    f"Total GRU Parameters  (3 gates): {gru_params_analytical} (PyTorch"
    f" verified: {gru_single_bias_params})"
)
print()

# ------------------------------------------------------------------------------
# Part (d): Gradient Clipping (Value vs Norm)
# ------------------------------------------------------------------------------
g_vec = np.array([3.0, 4.0, 12.0])
clip_threshold = 5.0

# 1. Clipping by value
g_clipped_val = np.clip(g_vec, -clip_threshold, clip_threshold)

# 2. Clipping by norm
norm_g = np.linalg.norm(g_vec)
g_clipped_norm = (
    g_vec * (clip_threshold / norm_g) if norm_g > clip_threshold else g_vec
)

# Verify directions using cosine similarity
cos_sim_val = np.dot(g_vec, g_clipped_val) / (
    np.linalg.norm(g_vec) * np.linalg.norm(g_clipped_val)
)
cos_sim_norm = np.dot(g_vec, g_clipped_norm) / (
    np.linalg.norm(g_vec) * np.linalg.norm(g_clipped_norm)
)

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (d): GRADIENT CLIPPING MECHANICS")
print("=" * 75,f"{RESET}")
print(f"Original Gradient vector       : {g_vec.tolist()} (L2-norm = {norm_g:.1f})")
print(f"Clipping by Value (threshold 5): {g_clipped_val.tolist()}")
print(f"Clipping by Norm  (max-norm 5) : {[round(x, 4) for x in g_clipped_norm.tolist()]}")
print(f"Cosine Similarity (Original vs Value-clipped) : {cos_sim_val:.4f} (Direction changed)")
print(f"Cosine Similarity (Original vs Norm-clipped)  : {cos_sim_norm:.4f} (Direction perfectly preserved)")
print("Addressed Problem: Exploding Gradients (prevents loss divergence and NaN crashes)")



PART (a): LSTM COMPUTATION AT TIME STEP t
New Cell State (c_t)   : 1.9000 (Expected: 1.9000)
New Hidden State (h_t) : 0.669366 (Expected: 0.668971)



PART (b): GRU COMPUTATION AT TIME STEP t
New GRU Hidden State (h_t) : 0.4400 (Expected: 0.4400)



PART (c): PARAMETER COUNTS (INPUT 8, HIDDEN 32)
Parameters per gate component : 1312
Total LSTM Parameters (4 gates): 5248 (PyTorch verified: 5248)
Total GRU Parameters  (3 gates): 3936 (PyTorch verified: 3936)



PART (d): GRADIENT CLIPPING MECHANICS
Original Gradient vector       : [3.0, 4.0, 12.0] (L2-norm = 13.0)
Clipping by Value (threshold 5): [3.0, 4.0, 5.0]
Clipping by Norm  (max-norm 5) : [1.1538, 1.5385, 4.6154]
Cosine Similarity (Original vs Value-clipped) : 0.9247 (Direction changed)
Cosine Similarity (Original vs Norm-clipped)  : 1.0000 (Direction perfectly preserved)
Addressed Problem: Exploding Gradients (prevents loss divergence and NaN crashes)
